In [29]:
!head -n 2 "agendamentos_30072026 - Página1 (1).csv"

id,paciente_id,profissional_id,procedimento_id,data_agendamento,hora_inicio,hora_fim,status,recorrencia,nome_paciente,nome_procedimento,nome_profissional,valor_procedimento,nome_convenio,pago,data_pagamento
ec2c0f73-0331-471f-b998-f5acc6279898,c52dc652-16d7-4cf2-b47a-9e2b5bd478ab,70b53c12-6e6c-48f7-b7be-4ffbcbf3372b,68c17a0e-b036-46a3-8578-1a0c29647104,2026-08-12,07:00:00,08:00:00,falta,semanal,Ruan Bombardi Benites,"TREINI (R$285,00)",Loraine Evelyn Larrea Ferreira Lescano,285.00,Judicialização,FALSE,null


In [30]:
# train_nn.py
"""
Treinamento da Rede Neural para previsão de Cancelamento (No-Show por cancelamento).
Lê um arquivo CSV com a estrutura da tabela agendamentos,
replica a engenharia de features e treina a rede.

Mudanças nesta versão (2026-07-30):
- Fonte de dados: CSV 'agendamentos_30072026.csv'.
- Alvo: CANCELAMENTO (status "desmarcado"/"cancelado"), pois a clínica perde receita em cancelamentos.
- Features redefinidas:
  - tem_pacote : baseado na coluna 'recorrencia' (diaria, mensal, semanal, quinzenal).
  - eh_bonus   : valor_procedimento é 0.00 ou null.
  - eh_evasao  : nome do procedimento contém "evas".
- Limpeza do valor_procedimento: remove separador de milhar (.) e troca vírgula por ponto.
- Log detalhado em cada etapa.
"""

import pandas as pd
import numpy as np
import joblib
import json
import logging
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# ================================================================
# CONFIGURAÇÕES
# ================================================================
STATUS_FALTA = ["desmarcado", "cancelado"]          # perda de receita
STATUS_PRESENCA = ["atendido", "confirmado", "falta"] # "falta" = pagou

COL_PACIENTE = "paciente_id"
COL_DATA = "data_agendamento"
COL_HORA = "hora_inicio"
COL_STATUS = "status"
COL_PROCEDIMENTO = "nome_procedimento"
COL_VALOR = "valor_procedimento"
COL_CONVENIO = "nome_convenio"
COL_RECORRENCIA = "recorrencia"

# ================================================================
# CARREGAMENTO COM DETECÇÃO AUTOMÁTICA DO SEPARADOR
# ================================================================

# Leitura simples: separador vírgula (padrão), aspas tratadas automaticamente
df = pd.read_csv(CAMINHO_CSV, sep=',', low_memory=False)
logger.info(f"Shape inicial: {df.shape}")

# Remove colunas inteiramente vazias (Unnamed) que sobraram da exportação
df = df.dropna(axis=1, how='all')
logger.info(f"Shape após remover colunas vazias: {df.shape}")

# Mantém apenas as 16 colunas de interesse (evita lixo)
colunas_esperadas = [
    "id", "paciente_id", "profissional_id", "procedimento_id",
    "data_agendamento", "hora_inicio", "hora_fim", "status",
    "recorrencia", "nome_paciente", "nome_procedimento", "nome_profissional",
    "valor_procedimento", "nome_convenio", "pago", "data_pagamento"
]
colunas_presentes = [c for c in colunas_esperadas if c in df.columns]
if not colunas_presentes:
    logger.error(f"Nenhuma coluna esperada encontrada. Colunas reais: {df.columns.tolist()}")
    raise ValueError("Estrutura do CSV não reconhecida.")

df = df[colunas_presentes]
logger.info(f"Colunas mantidas: {colunas_presentes}")
# ================================================================
# 1. LIMPEZA E VARIÁVEL ALVO (CANCELAMENTO)
# ================================================================
logger.info("Convertendo datas e horas...")
df[COL_DATA] = pd.to_datetime(df[COL_DATA], errors="coerce")
try:
    df["hora"] = pd.to_datetime(df[COL_HORA], format="%H:%M:%S", errors="coerce").dt.hour
except Exception:
    df["hora"] = df[COL_HORA].apply(lambda x: x.hour if hasattr(x, 'hour') else pd.NaT)

df["data_hora"] = pd.to_datetime(
    df[COL_DATA].astype(str) + " " + df[COL_HORA].astype(str),
    errors="coerce"
)

# Alvo: 1 = cancelamento, 0 = comparecimento (inclui falta)
logger.info(f"Alvo: Cancelamento (1) para status em {STATUS_FALTA}, 0 para {STATUS_PRESENCA}")
def classificar_desfecho(status):
    s = str(status).strip().lower()
    if s in STATUS_FALTA:
        return 1
    elif s in STATUS_PRESENCA:
        return 0
    else:
        return np.nan

df["Cancelamento"] = df[COL_STATUS].apply(classificar_desfecho)
antes = len(df)
df = df.dropna(subset=["Cancelamento"]).copy()
logger.info(f"Registros removidos (status não conclusivo): {antes - len(df)}")
logger.info(f"Total após filtro: {len(df)} linhas")
logger.info(f"Distribuição Cancelamento:\n{df['Cancelamento'].value_counts().to_dict()}")

# Convênios raros
LIMIAR = 30
conv_vol = df[COL_CONVENIO].value_counts()
raros = conv_vol[conv_vol < LIMIAR].index.tolist()
logger.info(f"Convênios com < {LIMIAR} atendimentos (removidos): {raros}")
df = df[~df[COL_CONVENIO].isin(raros)].copy()
logger.info(f"Linhas após remoção de convênios raros: {len(df)}")

# Limpeza e conversão do valor_procedimento
def limpar_valor(val):
    """Remove separador de milhar (.) e troca vírgula decimal por ponto."""
    if isinstance(val, str):
        val = val.strip()
        if val == '':
            return np.nan
        val = val.replace('.', '')          # remove pontos (milhar)
        val = val.replace(',', '.')         # vírgula -> ponto decimal
        try:
            return float(val)
        except ValueError:
            return np.nan
    else:
        try:
            return float(val)
        except (ValueError, TypeError):
            return np.nan

df[COL_VALOR] = df[COL_VALOR].apply(limpar_valor)
df["valor_servico"] = df[COL_VALOR].fillna(0).astype(float)
logger.info(f"Valor serviço: min={df['valor_servico'].min()}, max={df['valor_servico'].max()}, mediana={df['valor_servico'].median()}")

# Features baseadas em regras de negócio
logger.info("Criando features de pacote, bônus e evasão...")

# Pacote: definido pela recorrência
recorrencia_lower = df[COL_RECORRENCIA].astype(str).str.strip().str.lower()
df["tem_pacote"] = recorrencia_lower.isin(["diaria", "mensal", "semanal", "quinzenal"]).astype(int)
logger.info(f"Registros com tem_pacote=1: {df['tem_pacote'].sum()}")

# Bônus: valor do procedimento zerado ou nulo
df["eh_bonus"] = (df["valor_servico"] == 0).astype(int)
logger.info(f"Registros com eh_bonus=1: {df['eh_bonus'].sum()}")

# Evasão: palavra "evas" no procedimento
procedimento_lower = df[COL_PROCEDIMENTO].astype(str).str.lower()
df["eh_evasao"] = procedimento_lower.str.contains("evas", na=False).astype(int)
logger.info(f"Registros com eh_evasao=1: {df['eh_evasao'].sum()}")

# Faixa horária
df["faixa_horaria"] = pd.cut(
    df["hora"],
    bins=[-0.1, 4.9, 11.9, 17.9, 23.9],
    labels=["Madrugada", "Manhã", "Tarde", "Noite"]
)
logger.info("Faixa horária criada.")

# Dia da semana
df["dia_semana"] = df[COL_DATA].dt.day_name().map({
    "Monday": "Segunda", "Tuesday": "Terça", "Wednesday": "Quarta",
    "Thursday": "Quinta", "Friday": "Sexta", "Saturday": "Sábado", "Sunday": "Domingo"
})
logger.info("Dia da semana criado.")

# ================================================================
# 2. FEATURES HISTÓRICAS (shift)
# ================================================================
logger.info("Construindo features históricas (ordenando por paciente e data)...")
df = df.sort_values([COL_PACIENTE, "data_hora"]).reset_index(drop=True)

df["patient_total_appointments_past"] = df.groupby(COL_PACIENTE).cumcount()
df["patient_noshow_count_past"] = (
    df.groupby(COL_PACIENTE)["Cancelamento"]
    .transform(lambda s: s.shift(1).fillna(0).cumsum())
)
df["patient_noshow_rate_past"] = np.where(
    df["patient_total_appointments_past"] > 0,
    df["patient_noshow_count_past"] / df["patient_total_appointments_past"],
    0
)

def rolling_30d(group):
    group = group.sort_values("data_hora")
    result = (
        group.set_index("data_hora")["Cancelamento"]
        .rolling("30D").sum().shift(1).fillna(0)
    )
    result.index = group.index
    return result

logger.info("Calculando cancelamentos nos últimos 30 dias...")
df["patient_noshow_last_30d"] = (
    df.groupby(COL_PACIENTE).apply(rolling_30d).reset_index(level=0, drop=True)
)

def streak(group):
    group = group.sort_values("data_hora").copy()
    s = 0
    streaks = []
    for _, row in group.iterrows():
        streaks.append(s)
        s = s + 1 if row["Cancelamento"] == 1 else 0
    group["patient_noshow_streak"] = streaks
    return group

logger.info("Calculando streak de cancelamentos consecutivos...")
df = df.groupby(COL_PACIENTE, group_keys=False).apply(streak)

# ================================================================
# 3. SELEÇÃO DE FEATURES E ONE-HOT ENCODING
# ================================================================
features_num = [
    "valor_servico",
    "tem_pacote", "eh_bonus", "eh_evasao",
    "patient_total_appointments_past", "patient_noshow_rate_past",
    "patient_noshow_last_30d", "patient_noshow_streak"
]
features_cat = ["dia_semana", "faixa_horaria", COL_CONVENIO]

logger.info(f"Features numéricas: {features_num}")
logger.info(f"Features categóricas: {features_cat}")

df = df.dropna(subset=features_cat).copy()
X_raw = df[features_num + features_cat]
y = df["Cancelamento"]

logger.info(f"Shape de X_raw: {X_raw.shape}")
X = pd.get_dummies(X_raw, columns=features_cat, drop_first=True).astype(float)
logger.info(f"Shape após one-hot encoding: {X.shape}")

# ================================================================
# 4. SPLIT TREINO/TESTE
# ================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
logger.info(f"Treino: {X_train.shape[0]} amostras, Teste: {X_test.shape[0]} amostras")
logger.info(f"Distribuição no treino: {y_train.value_counts().to_dict()}")
logger.info(f"Distribuição no teste: {y_test.value_counts().to_dict()}")

# ================================================================
# 5. NORMALIZAÇÃO E PESOS DE CLASSE
# ================================================================
logger.info("Aplicando StandardScaler...")
scaler = StandardScaler()
X_train_nn = scaler.fit_transform(X_train)
X_test_nn = scaler.transform(X_test)

classes = np.array([0, 1])
pesos = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = {0: pesos[0], 1: pesos[1]}
logger.info(f"Pesos de classe: {class_weight}")

# ================================================================
# 6. ARQUITETURA DA REDE NEURAL
# ================================================================
logger.info("Construindo arquitetura da rede neural...")
tf.random.set_seed(42)
n_features = X_train_nn.shape[1]

model = keras.Sequential([
    keras.layers.Dense(16, activation="relu", input_shape=(n_features,)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)
model.summary(print_fn=logger.info)

# ================================================================
# 7. TREINAMENTO COM EARLY STOPPING
# ================================================================
logger.info("Iniciando treinamento com early stopping...")
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train_nn, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.15,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

ultima_epoca = len(history.history["loss"])
melhor_val_loss = min(history.history["val_loss"])
melhor_auc = max(history.history["val_auc"])
logger.info(f"Treinamento concluído em {ultima_epoca} épocas.")
logger.info(f"Melhor val_loss: {melhor_val_loss:.4f}, melhor val_auc: {melhor_auc:.4f}")

# ================================================================
# 8. SALVAMENTO DOS ARTEFATOS
# ================================================================
model.save("modelo_rede.keras")
joblib.dump(scaler, "scaler.pkl")
with open("colunas_modelo.json", "w") as f:
    json.dump(X_train.columns.tolist(), f)

cats_dict = {col: X_raw[col].unique().tolist() for col in features_cat}
with open("categorias_originais.json", "w") as f:
    json.dump(cats_dict, f)

logger.info("Artefatos salvos: modelo_rede.keras, scaler.pkl, colunas_modelo.json, categorias_originais.json")

/tmp/ipykernel_1513/3974336241.py:197: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby(COL_PACIENTE).apply(rolling_30d).reset_index(level=0, drop=True)
/tmp/ipykernel_1513/3974336241.py:211: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(COL_PACIENTE, group_keys=False).apply(streak)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass a

Epoch 1/200
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6238 - auc: 0.7312 - loss: 0.6083 - val_accuracy: 0.6653 - val_auc: 0.8169 - val_loss: 0.5345
Epoch 2/200
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6759 - auc: 0.8056 - loss: 0.5379 - val_accuracy: 0.7011 - val_auc: 0.8268 - val_loss: 0.4996
Epoch 3/200
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6878 - auc: 0.8196 - loss: 0.5214 - val_accuracy: 0.7025 - val_auc: 0.8329 - val_loss: 0.4954
Epoch 4/200
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6825 - auc: 0.8260 - loss: 0.5128 - val_accuracy: 0.6985 - val_auc: 0.8365 - val_loss: 0.4987
Epoch 5/200
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6903 - auc: 0.8293 - loss: 0.5136 - val_accuracy: 0.6979 - val_auc: 0.8375 - val_loss: 0.4927
Epoch 6/200
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.6870 - auc: 0.8355 - loss: 0.5024 - val_accuracy: 0.7078 - val_auc: 0.8382 - val_loss: 0.4830
Epoch 7/200
1008/1008 

In [31]:
from sklearn.metrics import precision_recall_curve

# Probabilidades no teste (já calculadas após o treino)
prob_nn = model.predict(X_test_nn, verbose=0).ravel()

# Define o recall alvo (ex.: 0.90 = capturar 90% dos cancelamentos)
RECALL_ALVO = 0.90

precisions, recalls, thresholds = precision_recall_curve(y_test, prob_nn)
# Alinhar tamanhos (precision_recall_curve retorna 1 a mais)
precisions, recalls = precisions[:-1], recalls[:-1]

# Encontrar o menor threshold que atinge recall >= alvo
idx_ok = np.where(recalls >= RECALL_ALVO)[0]
if len(idx_ok) > 0:
    # Entre os que atendem, escolher o de maior precision
    idx = idx_ok[np.argmax(precisions[idx_ok])]
    threshold_escolhido = thresholds[idx]
else:
    threshold_escolhido = 0.5  # fallback

print(f"Threshold escolhido (recall ≥ {RECALL_ALVO}): {threshold_escolhido:.4f}")

Threshold escolhido (recall ≥ 0.9): 0.3623
